In [1]:
import fitz  # PyMuPDF
import json

def extract_structured_text(pdf_path):
    """
    提取PDF的结构化文本（标题、段落、列表、表格等）
    :param pdf_path: PDF文件路径
    :return: 结构化字典（可转为JSON）
    """
    doc = fitz.open(pdf_path)
    structured_data = {"metadata": {}, "sections": []}
    
    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        blocks = page.get_text("dict")["blocks"]  # 获取文本块（包含位置和样式信息）
        
        current_section = {"title": "", "content": [], "tables": []}
        
        for block in blocks:
            # 处理文本块
            if "lines" in block:
                for line in block["lines"]:
                    for span in line["spans"]:
                        text = span["text"].strip()
                        font_size = span["size"]
                        is_bold = "bold" in span["font"].lower()
                        
                        # 推断标题（根据字体大小和加粗）
                        if is_bold and font_size >= 12:  # 医学文档通常H1标题>12pt
                            if current_section["content"]:  # 保存上一个章节
                                structured_data["sections"].append(current_section.copy())
                                current_section = {"title": text, "content": [], "tables": []}
                            else:
                                current_section["title"] = text
                        else:
                            if text:
                                current_section["content"].append(text)
                    
        if current_section["content"] or current_section["tables"]:
            structured_data["sections"].append(current_section)
    
    # 提取元数据（如文档标题、作者）
    structured_data["metadata"] = {
        "title": doc.metadata.get("title", ""),
        "author": doc.metadata.get("author", ""),
        "pages": len(doc)
    }
    
    doc.close()
    return structured_data

# 示例使用
pdf_path = "/home/wangjun/study/rag/wj-miaoshou/ratubrain-semiconductor/app/data/raw/专家共识与指南/1½σ.pdf"
structured_text = extract_structured_text(pdf_path)

# 输出为JSON文件
with open("output_structured.json", "w", encoding="utf-8") as f:
    json.dump(structured_text, f, indent=2, ensure_ascii=False)

# 打印部分结果预览（第一个章节）
print("==== 提取结果预览 ====")
first_section = structured_text["sections"][0] if structured_text["sections"] else {}
print(f"标题: {first_section.get('title', '无')}")
print("内容前两段:")
print("\n".join(first_section.get("content", [])[:2]))
print(f"表格数量: {len(first_section.get('tables', []))}")


==== 提取结果预览 ====
标题: 
内容前两段:
ICS
**.***.**
表格数量: 0


In [2]:
import sys
import os

# 获取当前 Notebook 的路径
notebook_dir = os.getcwd()

# 添加项目根目录到 sys.path
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))  # 上一级目录
sys.path.append(project_root)

# 现在可以正常导入
from rag.data_processor import SimpleMedicalProcessor


/home/wangjun/miniconda3/envs/rag/lib/python3.9/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
process = SimpleMedicalProcessor()
documents = process.load_documents("./test")


🔍 递归扫描目录: ./test
📚 找到 5 个MD文件
✅ 成功加载 5 个医学文档


In [4]:
import json
from langchain.schema import Document
chunks = process.smart_chunk_documents(documents)
def save_chunks_to_json(chunks, file_path: str):
    """将分块结果保存为结构化JSON文件"""
    # 转换为可序列化的字典列表
    output_data = []
    for i, chunk in enumerate(chunks):
        chunk_data = {
            "id": i + 1,
            "content": chunk.page_content,
            "metadata": dict(chunk.metadata),
            "length": len(chunk.page_content),
            "source": chunk.metadata.get("source", "unknown"),
            "path": chunk.metadata.get("relative_path", "unknown"),
        }
        output_data.append(chunk_data)
    
    # 保存到JSON文件
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=2, ensure_ascii=False)
    
    print(f"✅ 已保存 {len(output_data)} 个分块到 {file_path}")
save_chunks_to_json(chunks, "output_chunks.json")

处理文档: 2.《镇静催眠药物所致精神和行为障碍中医神志病临床诊疗指南》 公示稿.docx.md
24 个标题分块
属于{'Header 1': '中华中医药学会标准T/CACM xxx-202x'} 
 标题层级: 中医神志病诊疗指南镇静催眠药物所致的精神  
和行为障碍  
The Tradtional Chin
属于{'Header 1': '目次'} 
 标题层级: 前言.  
引言..
1范围...
2规范性引用文件.
3 术语和定义
4诊断与鉴别诊断..4.1诊
属于{'Header 1': '前言'} 
 标题层级: 本标准按照GB/T1.1-2009给出的规则起草。
本标准由神志病分会提出。  
本标准由中华中医药
属于{'Header 1': '引言'} 
 标题层级: 国内外对于镇静催眠药物所致精神和行为障碍的中医诊断及干预治疗仍在发展，新的诊断与治疗方法也在不断出现
属于{'Header 1': '1. 范围'} 
 标题层级: 本《指南》规定了镇静催眠药物所致精神和行为障碍的术语和定义、特点、评判标准、中医药干预原则及中医药干
属于{'Header 1': '2. 规范性引用文件'} 
 标题层级: 下列文件对于本文件的应用是必不可少的。凡是注日期的引用文件，仅注日期的版本适用于本文件。凡是不注日期
属于{'Header 1': '3. 术语和定义[1-3]'} 
 标题层级: 下列术语和定义适用于本指南。  
3.1 镇静催眠药物 (sedative-hypnotic dru
属于{'Header 1': '4. 诊断与鉴别诊断[1,3,5.7-9,11,13]', 'Header 2': '4. 1诊断要点'} 
 标题层级: a)出现临床症状前有明确的服用镇静催眠药物病史。
b)出现临床症状与服用的镇静催眠药物的种类及剂量有
属于{'Header 1': '4. 1.1戒断状态'} 
 标题层级: 戒断状态是依赖综合征的指征之一，如果这些症状是就诊的原因或严重到足以引起医疗上的重视，则戒断状态应作
属于{'Header 1': '4. 1.2依赖综合症'} 
 标题层级: (1）对使用该物质的强烈渴望或冲动感;(2）对该物质使用行为的开始、结束及计量难以控制;(3）当该物
属于{'Header 1': '4. 1.2依赖综合症', 'H

In [5]:
import numpy as np
print(np.mean([len(chunk.page_content) for chunk in chunks]))

232.7906976744186
